# Phase 0 + 1 — Data exploration

**Goal:** Understand the OHS field visit data well enough to lock modeling decisions.

Each row = one **order** (or one visit with no order). A workplace can appear many times.

**Questions I'm answering:**
- What order types exist, and which ones mean "serious"?
- Is the target learnable at workplace level?
- Who gets a label vs who doesn't (selection bias)?
- Can I stack years 2017–2023 for modeling?


In [31]:
from pathlib import Path
import pandas as pd

ROOT = Path("..").resolve()
DATA = ROOT / "data"

def load_year(year):
    path = DATA / f"{year}_ohs_field_visit.xlsx"
    df = pd.read_excel(path)
    df["source_year"] = year
    return df


def is_blank_order(s):
    return s.isna()


def has_serious_order(s, serious_types):
    return s.isin(serious_types).any()

In [32]:
years = range(2020, 2025)
cols = {}
for y in years:
    cols[y] = list(pd.read_excel(DATA / f"{y}_ohs_field_visit.xlsx", nrows=0).columns)

for y in years:
    print(y, "== 2023?", cols[y] == cols[2023])

# Show diffs vs 2023
base = set(cols[2023])
for y in [2020, 2021, 2022, 2024]:
    print(f"\n{y} only:", set(cols[y]) - base)
    print(f"2023 only:", base - set(cols[y]))

2020 == 2023? True
2021 == 2023? False
2022 == 2023? False
2023 == 2023? True
2024 == 2023? False

2020 only: set()
2023 only: set()

2021 only: {'Unnamed: 0', 'FIELD VISIT ID', 'ORDER ID'}
2023 only: set()

2022 only: {'Unnamed: 0', 'FIELD VISIT ID', 'ORDER ID'}
2023 only: set()

2024 only: {'ACT-REGULATION NAME', 'ACT-REG ID', 'CASE ACT', 'CONTRAVENER_ORG ID'}
2023 only: {'ACT', 'CONTRAVENER - ORG ID', 'REG ID', 'REG NAME', 'REG YR'}


In [38]:
df24 = load_year(2024)
df23 = load_year(2023)
df24.info()

<class 'pandas.DataFrame'>
RangeIndex: 119990 entries, 0 to 119989
Data columns (total 22 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   FIELD VISIT DATE               119990 non-null  datetime64[us]
 1   FIELD VISIT TYPE               119990 non-null  str           
 2   CASE TYPE                      119990 non-null  str           
 3   CASE STATUS                    119990 non-null  str           
 4   WORKPLACE ID                   119982 non-null  float64       
 5   WORKPLACE NAME AT FV TIME      119987 non-null  object        
 6   WORKPLACE ADDRESSS AT FV TIME  119961 non-null  str           
 7   POSTAL CODE                    115003 non-null  object        
 8   PRIMARY NAICS                  119964 non-null  float64       
 9   NAICS DESCRIPTION              119964 non-null  str           
 10  CONTRAVENER ROLE               88436 non-null   str           
 11  CONTRAVENER

## 1. Basic counts — 2023 and 2024

How many rows, workplaces, and order types? ~29% of rows have no order (NaN) — that's normal; many visits find nothing to write up.

In [34]:
print("Rows:", len(df24))
print("Unique workplaces:", df24["WORKPLACE ID"].nunique())
print("\nORDER TYPE COUNTS")
print(df24["ORDER TYPE"].value_counts(dropna=False))

blank = is_blank_order(df24["ORDER TYPE"])
print(f"\nBlank/NULL orders (order_type): {blank.sum():,} ({blank.mean():.1%} of rows)")



Rows: 119082
Unique workplaces: 30911

ORDER TYPE COUNTS
ORDER TYPE
Time Based Order                 44466
NaN                              34257
Forthwith Order                  21264
Time Unknown Order                8104
Stop Use/Stop Work Order          5731
Requirement Time Based            4521
Plan Order                         353
Requirement Forthwith              194
BOSTA Refrain                       87
BOSTA Compliance Order-FORT         49
Requirement Time Unknown            26
BOSTA Demand-TIME                   15
BOSTA Compliance Order-TIME         11
BOSTA Compliance Assistance          2
BOSTA Notice Of Contravention        2
Name: count, dtype: int64

Blank/NULL orders (order_type): 34,257 (28.8% of rows)


In [35]:
print("Rows:", len(df23))
print("Unique workplaces:", df23["WORKPLACE ID"].nunique())
print("\nORDER TYPE COUNTS")
print(df23["ORDER TYPE"].value_counts(dropna=False))

blank = is_blank_order(df23["ORDER TYPE"])
print(f"\nBlank/NULL orders (order_type): {blank.sum():,} ({blank.mean():.1%} of rows)")

Rows: 119082
Unique workplaces: 30911

ORDER TYPE COUNTS
ORDER TYPE
Time Based Order                 44466
NaN                              34257
Forthwith Order                  21264
Time Unknown Order                8104
Stop Use/Stop Work Order          5731
Requirement Time Based            4521
Plan Order                         353
Requirement Forthwith              194
BOSTA Refrain                       87
BOSTA Compliance Order-FORT         49
Requirement Time Unknown            26
BOSTA Demand-TIME                   15
BOSTA Compliance Order-TIME         11
BOSTA Compliance Assistance          2
BOSTA Notice Of Contravention        2
Name: count, dtype: int64

Blank/NULL orders (order_type): 34,257 (28.8% of rows)


## 2. What counts as "serious"?

Per the data dictionary: **Stop Work** and **Time Unknown** orders signal immediate danger. Labels changed in 2024 (one bundled stop-work type → four subtypes), so we keep separate lists.

In [36]:
SERIOUS_2024 = {
    "Time unknown order",
    "STOP A057-6a: Compliance no visit required",
    "STOP A057-6b: Compliance re-inspect",
    "STOP A057-6c: Barrier",
    "STOP A057-8: WHMIS",
}
SERIOUS_2023 = {
    "Stop Use/Stop Work Order",   # 2023 rolls all stop types into one label
    "Time Unknown Order",
}

## 3. Workplace-level target rate (2023)

Aggregate to workplace: did this site get **any** serious order in 2023?

In [44]:
target_2023 = (
    df23.groupby("WORKPLACE ID")["ORDER TYPE"]
    .apply(lambda s: has_serious_order(s, SERIOUS_2023))
)
print(f"2023 workplaces: {len(target_2023):,}")
print(f"Serious-order rate: {round(100 *target_2023.mean(), 2)}",  "%")


2023 workplaces: 30,911
Serious-order rate: 9.27 %


### Who gets a label?

If I include every workplace ever seen in 2017–2022 but **not** visited in 2023, the positive rate drops to ~0.6%. Those zeros mostly mean "never inspected," not "safe." So the evaluation cohort = history **and** a 2023 visit.

In [45]:
# Load only what we need — full files are slow
COLS = ["WORKPLACE ID", "ORDER TYPE"]
hist = pd.concat([load_year(y)[COLS] for y in range(2017, 2023)])

hist_wps = set(hist["WORKPLACE ID"].unique())
wps_2023 = set(df23["WORKPLACE ID"].unique())
cohort = hist_wps & wps_2023

rate_all = target_2023.mean()
rate_cohort = target_2023.reindex(list(cohort), fill_value=False).mean()
rate_hist = target_2023.reindex(list(hist_wps), fill_value=False).mean()

print(f"All 2023 visits:        {rate_all:.1%}")
print(f"History + 2023 visit:   {rate_cohort:.1%}  <- use for modeling")
print(f"All history:            {rate_hist:.1%}  <- misleading (includes never inspected)")
print(f"\nCohort size: {len(cohort):,} workplaces")

KeyboardInterrupt: 

## 4. Case type mix — selection bias

~40% of 2023 visits are **Investigations** (reactive — after a complaint, injury, etc.). Labels only exist where the Ministry chose to go. That's the main bias risk; we name it in the deck, not try to fix it statistically in a weekend prototype.

In [ ]:
print(df23["CASE TYPE"].value_counts(normalize=True).mul(100).round(1))

## 5. Out-of-time split sanity check

2022 and 2023 serious-order rates are similar (~9%), so using features through 2022 and predicting 2023 is reasonable.

In [ ]:
df22 = load_year(2022)[COLS]
target_2022 = (
    df22.groupby("WORKPLACE ID")["ORDER TYPE"]
    .apply(lambda s: has_serious_order(s, SERIOUS_2023))
)
print(f"2022 serious-order rate: {target_2022.mean():.1%}")
print(f"2023 serious-order rate: {target_2023.mean():.1%}")

## Phase 0 — decisions locked

**Target (D004):** Workplace gets a Stop Work or Time Unknown order in 2023 → `target = 1`. Not fatalities, not "any order."

**Split (D005):** Features from 2017–2022, predict 2023. Out-of-time, not random.

**Cohort (D008):** Model and evaluate on ~12k workplaces that were visited before *and* in 2023 (~8% positive). Scoring the full backlog is the operational story; labels only exist where we inspected.

**Schema (D007):** Stack 2017–2023 for the prototype. 2024 needs a rename map before we use it.

**Bias (D006):** ~40% investigations, labels only on inspected sites. Acknowledge in Slide 5; recommend reserved exploratory inspection capacity.

**Next step:** Build the modeling table — one row per workplace in the evaluation cohort, ~10 feature columns, binary target.